---
# Denoising Diffusion Implicit Models (DDIM Practice)
---

In this exercise, we will implement **DDIM (Denoising Diffusion Implicit Models)** using our trained noise predictor on the AFHQ Cat dataset.

While DDPM requires iterating through all $T=1000$ Markov chain steps during generation, DDIM formulates a non-Markovian diffusion process that allows us to:
1. Sample **deterministically** when $\eta = 0$.
2. **Accelerate sampling** by skipping steps (e.g., generating crisp images in just 20 or 50 steps instead of 1000) using the exact same trained U-Net!

Paper: [Denoising Diffusion Implicit Models](https://arxiv.org/abs/2010.02502)

In [1]:
import importlib
import torch
import data, trainer, utils, visual, tests, models


---
## 1. DDIM Mathematical Formulation
---

Given a noisy image $x_t$ at step $t$ and our network's noise prediction $\epsilon_\theta(x_t, t)$, we first predict the denoised image $x_0^{\text{pred}}$:

$$
x_0^{\text{pred}} = \frac{x_t - \sqrt{1 - \bar{\alpha}_t}\,\epsilon_\theta(x_t, t)}{\sqrt{\bar{\alpha}_t}}
$$

To step backward to a previous time $t_{\text{prev}}$, the DDIM update equation is:

$$
x_{t_{\text{prev}}} = \sqrt{\bar{\alpha}_{t_{\text{prev}}}}\,x_0^{\text{pred}} + \sqrt{1 - \bar{\alpha}_{t_{\text{prev}}} - \sigma_t^2}\,\epsilon_\theta(x_t, t) + \sigma_t \varepsilon
$$

Where $\varepsilon \sim \mathcal{N}(0, I)$ and the variance $\sigma_t$ is controlled by hyperparameter $\eta$:

$$
\sigma_t = \eta \sqrt{\frac{1 - \bar{\alpha}_{t_{\text{prev}}}}{1 - \bar{\alpha}_t} \left(1 - \frac{\bar{\alpha}_t}{\bar{\alpha}_{t_{\text{prev}}}}\right)}
$$

* **When $\eta = 0$:** $\sigma_t = 0$, making generation **100% deterministic**.
* **When $\eta = 1$:** The step variance matches DDPM.

### **Task:**
Implement `get_ddim_timesteps`, `ddim_p_sample`, and `ddim_sample` in `utils.py`.

In [2]:
importlib.reload(tests)
importlib.reload(utils)

tests.test_get_ddim_timesteps()
tests.test_ddim_p_sample()
tests.test_ddim_sample()

✅ PASS: get_ddim_timesteps is correct!
✅ PASS: ddim_p_sample is correct!
✅ PASS: ddim_sample is correct!


True

---
## 2. Train the Noise Predictor
---
Because DDIM shares the exact same training loss as DDPM, we train our U-Net normally.

In [3]:
importlib.reload(trainer)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model, history = trainer.train_ddpm(
    num_timesteps=1000,
    base_channels=64,
    time_dim=256,
    groups=8,
    batch_size=64,
    lr=2e-4,
    betas=(0.9, 0.999),
    weight_decay=1e-4,
    epochs=50,
    device=device,
    num_workers=2,
    image_size=32
)

visual.show_training_stats(history, title="Model Training").show()

Using device: cuda:0


Extracting AFHQ v2:   0%|          | 0/15811 [00:00<?, ?file/s]

Creating resized AFHQ cache 32x32:   0%|          | 0/15803 [00:00<?, ?it/s]

AFHQ raw dir: /data/afhq_v2
AFHQ resized dir: /data/afhq_v2_32
AFHQ image size: 32x32
AFHQ cat train: 5065
AFHQ cat val: 246
AFHQ cat test: 247


Epochs:   0%|          | 0/50 [00:00<?, ?it/s]

---
## 3. Accelerated DDIM Sampling & Comparison
---
Let's see the power of DDIM by sampling the same model with only **50 steps** deterministically ($\eta = 0.0$).

In [4]:
model.eval()
schedule = utils.precompute_ddpm_schedule(1000, device=device)

# Sample in only 50 steps!
ddim_samples, ddim_trajectories = utils.ddim_sample(
    model, num_samples=16, image_size=32, num_timesteps=1000, 
    num_ddim_steps=50, device=device, schedule=schedule, eta=0.0
)

visual.animate_trajectories(ddim_trajectories, title="DDIM (50 Steps, eta=0)", savepath="ddim_50.gif", fps=10)
visual._image_grid(ddim_samples, title="DDIM 50-Step Generated Samples", cols=4).show()